In [19]:
# Setup project imports
from pathlib import Path
import sys

def find_project_root():
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "utils").exists() and (p / "models").exists():
            return p
    for p in [here, *here.parents]:
        if any((p / m).exists() for m in (".git", "requirements.txt", "pyproject.toml")):
            return p
    return here

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

%load_ext autoreload
%autoreload 2

Project root: e:\Automation_IT\3\Drift case study\MAIT_G2_Drift_Challenge
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn
import pandas as pd
import contextlib
import plotly.graph_objects as go

from utils.reproducibility_utils import seed_everything
from utils.data_loadingV2 import load_data, load_test_data, feature_names
from utils.LDS_FocalR_Loss import EnhancedLDS_FocalR_Loss
from utils.SimpleBucketLoss import SimpleBucketLoss
from utils.kaggle_metric import score
from models.lstm import LSTMModel

plt.style.use('seaborn-v0_8')

In [21]:
CONFIG = {
    # Paths
    'TRAIN_PATH': "../datasets_challenge/train",
    'TEST_PATH': "../datasets_challenge/test",
    'DIST_CSV': "../datasets_challenge/sideSlip_distribution.csv",
    'CHECKPOINT_NAME': "best_modelv6.pth",
    'SUBMISSION_CSV': "predictionsv6.csv",
    
    # Reproducibility
    'SEED': 42,
    
    # Data Loading
    'TRAIN_RATIO': 0.8,
    'BATCH_SIZE': 512,  # Increased back to 512 for more stable gradients
    'SEQUENCE_LENGTH': 20,
    'THRESHOLD': True,
    'THRESHOLD_VALUE': 3.0,
    'THRESHOLD_COLUMN': "vxCG",
    'COLUMNS_TO_DROP': ["sideSlip", "vyCG", "time", "longitude", "latitude"],
    'TARGET': "sideSlip",
    'SCALING': True,
    
    # Model Architecture
    'MODEL_TYPE': 'LSTM',
    'RNN_UNITS': 128,
    'FC_UNITS': 128,
    'NUM_LAYERS': 2,
    'DROPOUT': [0.3, 0.2, 0.1],
    'USE_BATCH_NORM': True,
    
    # Loss Configuration
    'LOSS_TYPE': 'EnhancedLDS_FocalR_Loss',
    'LOSS_WEIGHTS': (1.0, 3.0, 8.0),  
    'USE_HUBER': True,
    'HUBER_BETA': 0.08,
    'LDS_SIGMA': 2.0,   
    'FOCAL_GAMMA': 1.0,  
    'BINS': 100,
    
    # Optimizer
    'LEARNING_RATE': 3e-4,  
    'WEIGHT_DECAY': 6e-4,  
    'BETAS': (0.9, 0.999),
    'MAX_GRAD_NORM': 0.7,
    
    # Scheduler
    'USE_SCHEDULER': True,
    'SCHEDULER_T_MAX': 300,  
    'SCHEDULER_ETA_MIN': 5e-7,  
    
    # Training
    'NUM_EPOCHS': 300,
    'EARLY_STOP_PATIENCE': 100,  # Critical fix: Actually enable early stopping!
    'USE_AMP': True,
    'EMA_DECAY': 0.996,  # Slightly increased for more stable EMA
    
    # Challenge Constraints
    'MANY_SHOT_CONSTRAINT': 0.2,
    'FEW_SHOT_TARGET': 0.3,
    
    # Visualization
    'FIG_SIZE': (14, 8),
    'PLOTLY_WIDTH': 1200,
    'PLOTLY_HEIGHT': 500,
}

print("Configuration loaded successfully")

Configuration loaded successfully


In [22]:
seed_everything(CONFIG['SEED'])
print(f"Random seed set to {CONFIG['SEED']}")

Random seed set to 42


## Load Data

In [23]:
train_dataloader, val_dataloader, pipeline, mean_dict, std_dict = load_data(
    train_path=CONFIG['TRAIN_PATH'],
    train_ratio=CONFIG['TRAIN_RATIO'],
    batchsize=CONFIG['BATCH_SIZE'],
    columns_to_standardize=feature_names,
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold=CONFIG['THRESHOLD'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    scaling=CONFIG['SCALING'],
    seed=CONFIG['SEED'],
    shuffle=True
)

print(f"\nTraining batches: {len(train_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")
print(f"Feature count: {len(feature_names)}")
print(f"\n📊 Scaling Statistics (mean/std):")
for feat in feature_names[:5]:
    print(f"  {feat:20s}: mean={mean_dict.get(feat, 0.0):8.4f}, std={std_dict.get(feat, 1.0):8.4f}")

../datasets_challenge/train\train_part_001.csv
../datasets_challenge/train\train_part_002.csv
../datasets_challenge/train\train_part_003.csv
../datasets_challenge/train\train_part_004.csv
../datasets_challenge/train\train_part_005.csv
../datasets_challenge/train\train_part_006.csv
../datasets_challenge/train\train_part_007.csv
../datasets_challenge/train\train_part_008.csv
../datasets_challenge/train\train_part_009.csv
../datasets_challenge/train\train_part_010.csv
../datasets_challenge/train\train_part_011.csv
../datasets_challenge/train\train_part_012.csv
../datasets_challenge/train\train_part_013.csv
../datasets_challenge/train\train_part_014.csv
../datasets_challenge/train\train_part_015.csv
../datasets_challenge/train\train_part_016.csv
../datasets_challenge/train\train_part_017.csv
../datasets_challenge/train\train_part_018.csv
../datasets_challenge/train\train_part_019.csv
../datasets_challenge/train\train_part_020.csv
../datasets_challenge/train\train_part_021.csv
../datasets_c

In [24]:
# Extract training targets for loss function
all_train_targets = []
for i in range(len(train_dataloader.dataset)):
    _, y = train_dataloader.dataset[i]
    all_train_targets.append(y.item())

y_train_np = np.array(all_train_targets)
print(f"Training samples: {len(y_train_np):,}")

Training samples: 104,070


## Setup Model & Training

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# AMP setup
if torch.cuda.is_available():
    amp_device_type = "cuda"
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler(amp_device_type, enabled=CONFIG['USE_AMP'])
else:
    amp_device_type = None
    amp_dtype = None
    scaler = torch.amp.GradScaler("cuda", enabled=False)

def autocast_ctx():
    if amp_device_type is None or not CONFIG['USE_AMP']:
        return contextlib.nullcontext()
    return torch.amp.autocast(device_type=amp_device_type, dtype=amp_dtype, enabled=True)

# Get input size
X_sample, _ = next(iter(train_dataloader))
input_size = X_sample.shape[2]
seq_length = X_sample.shape[1]

# Create model
model = LSTMModel(
    input_size=input_size,
    output_size=1,
    rnn_units=CONFIG['RNN_UNITS'],
    fc_units=CONFIG['FC_UNITS'],
    dropout=CONFIG['DROPOUT'],
    num_layers=CONFIG['NUM_LAYERS'],
    use_batch_norm=CONFIG['USE_BATCH_NORM']
).to(device)

# Create loss function
if CONFIG['LOSS_TYPE'] == 'SimpleBucket':
    criterion = SimpleBucketLoss(
        train_targets=y_train_np,
        bucket_weights=CONFIG['LOSS_WEIGHTS'],
        use_huber=CONFIG['USE_HUBER'],
        huber_beta=CONFIG['HUBER_BETA'],
        dist_csv_path=CONFIG['DIST_CSV']
    ).to(device)
else:
    criterion = EnhancedLDS_FocalR_Loss(
        train_targets=y_train_np,
        bucket_weights=CONFIG['LOSS_WEIGHTS'],
        lds_kernel='gaussian',
        lds_ks=CONFIG['LDS_SIGMA'],
        lds_sigma=CONFIG['LDS_SIGMA'],
        focal_gamma=CONFIG['FOCAL_GAMMA'],
        use_huber=CONFIG['USE_HUBER'],
        huber_beta=CONFIG['HUBER_BETA'],
        num_bins=CONFIG['BINS'],
        dist_csv_path=CONFIG['DIST_CSV']
    ).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

Device: cpu

[EnhancedLDS+FocalR Loss - CSV-Aligned Configuration]
Loaded 234 band definitions from CSV
Approximate boundaries (max |sideslip| per band):
  Many-shot:   <= 2.4000 degrees
  Medium-shot: <= 5.0000 degrees
Data distribution:
  Many:    95248 (91.52%)
  Medium:   7024 ( 6.75%)
  Few:      1798 ( 1.73%)
Bucket weights: 1.0 / 3.0 / 8.0
Focal gamma: 1.0 | Huber beta: 0.08
LDS: kernel=gaussian, sigma=2.0

Model parameters: 235,053


In [26]:
# Optimizer & Scheduler
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['LEARNING_RATE'],
    weight_decay=CONFIG['WEIGHT_DECAY'],
    betas=CONFIG['BETAS']
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG['SCHEDULER_T_MAX'],
    eta_min=CONFIG['SCHEDULER_ETA_MIN']
) if CONFIG['USE_SCHEDULER'] else None

# EMA
ema_params = [p.detach().clone() for p in model.parameters()]

def ema_update(model_params, ema_params, decay):
    for p, e in zip(model_params, ema_params):
        e.mul_(decay).add_(p.detach(), alpha=1.0 - decay)

@torch.no_grad()
def load_params_(model_params, src_params):
    for p, s in zip(model_params, src_params):
        p.copy_(s)

# Training state
train_curve, val_curve = [], []
train_mae_curve, val_mae_curve = [], []
bucket_many, bucket_medium, bucket_few = [], [], []
learning_rate_curve = []
best_loss = float("inf")
best_epoch = -1
epochs_no_improve = 0

print("Training setup complete")

Training setup complete


## Training Loop

In [27]:
for epoch in range(CONFIG['NUM_EPOCHS']):
    # Training
    model.train()
    running_loss, running_mae = 0.0, 0.0
    num_train_batches = len(train_dataloader)

    for xb, yb in train_dataloader:
        xb = xb.to(device, non_blocking=True).float()
        yb = yb.to(device, non_blocking=True).float().reshape(-1, 1)

        optimizer.zero_grad(set_to_none=True)
        
        with autocast_ctx():
            pred = model(xb)
            total_loss = criterion(pred, yb)
        
        scaler.scale(total_loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['MAX_GRAD_NORM'])
        scaler.step(optimizer)
        scaler.update()
        
        ema_update(model.parameters(), ema_params, CONFIG['EMA_DECAY'])
        
        running_loss += total_loss.item()
        running_mae += (pred - yb).abs().mean().item()

    avg_train_loss = running_loss / num_train_batches
    avg_train_mae = running_mae / num_train_batches
    train_curve.append(avg_train_loss)
    train_mae_curve.append(avg_train_mae)

    # Validation
    model.eval()
    
    def _evaluate(current_params):
        snapshot = [p.detach().clone() for p in model.parameters()]
        load_params_(model.parameters(), current_params)
        
        running_vloss, running_vmae = 0.0, 0.0
        all_val_preds, all_val_targets = [], []
        num_val_batches = len(val_dataloader)
        
        with torch.no_grad():
            for xv, yv in val_dataloader:
                xv = xv.to(device, non_blocking=True).float()
                yv = yv.to(device, non_blocking=True).float().reshape(-1, 1)
                
                with autocast_ctx():
                    pred_val = model(xv)
                    total_loss_val = criterion(pred_val, yv)
                
                running_vloss += total_loss_val.item()
                running_vmae += (pred_val - yv).abs().mean().item()
                all_val_preds.append(pred_val.cpu())
                all_val_targets.append(yv.cpu())
        
        all_val_preds = torch.cat(all_val_preds, dim=0).to(device)
        all_val_targets = torch.cat(all_val_targets, dim=0).to(device)
        bucket_stats = criterion.bucket_metrics(all_val_preds, all_val_targets)
        avg_vloss = running_vloss / num_val_batches
        avg_vmae = running_vmae / num_val_batches
        
        load_params_(model.parameters(), snapshot)
        return avg_vloss, avg_vmae, bucket_stats
    
    # Evaluate both live and EMA
    live_params = [p.detach().clone() for p in model.parameters()]
    vloss_live, vmae_live, buckets_live = _evaluate(live_params)
    vloss_ema, vmae_ema, buckets_ema = _evaluate(ema_params)
    
    if vloss_ema <= vloss_live:
        best_view = "EMA"
        best_vloss, best_vmae = vloss_ema, vmae_ema
        best_buckets = buckets_ema
        best_params = [p.detach().clone() for p in ema_params]
    else:
        best_view = "LIVE"
        best_vloss, best_vmae = vloss_live, vmae_live
        best_buckets = buckets_live
        best_params = live_params
    
    val_curve.append(best_vloss)
    val_mae_curve.append(best_vmae)
    
    many_mae = best_buckets['mae_many']
    med_mae = best_buckets['mae_medium']
    few_mae = best_buckets['mae_few']
    
    bucket_many.append(many_mae)
    bucket_medium.append(med_mae)
    bucket_few.append(few_mae)
    
    current_lr = optimizer.param_groups[0]['lr']
    learning_rate_curve.append(current_lr)
    if scheduler is not None:
        scheduler.step()

    # Logging
    constraint_symbol = "🎯" if many_mae <= CONFIG['MANY_SHOT_CONSTRAINT'] else ""
    print(f"Epoch {epoch+1:3d}/{CONFIG['NUM_EPOCHS']} | "
          f"TrL: {avg_train_loss:.5f} | VaL: {best_vloss:.5f} | "
          f"TrMAE: {avg_train_mae:.5f} | VaMAE: {best_vmae:.5f} | "
          f"Many: {many_mae:.4f}{constraint_symbol} | Med: {med_mae:.4f} | Few: {few_mae:.4f} | "
          f"LR: {current_lr:.6f} | {best_view}", end="")

    # Checkpointing
    passes_constraint = many_mae <= CONFIG['MANY_SHOT_CONSTRAINT']
    
    if best_epoch == -1:
        improved = True
    elif passes_constraint:
        prev_passes = bucket_many[best_epoch] <= CONFIG['MANY_SHOT_CONSTRAINT']
        improved = (not prev_passes) or (prev_passes and best_vloss < best_loss - 1e-9)
    else:
        prev_passes = bucket_many[best_epoch] <= CONFIG['MANY_SHOT_CONSTRAINT']
        improved = (not prev_passes) and (best_vloss < best_loss - 1e-9)
    
    if improved:
        best_loss = best_vloss
        best_epoch = epoch
        epochs_no_improve = 0
        
        checkpoint_dict = {
            'epoch': epoch,
            'model_type': CONFIG['MODEL_TYPE'],
            'model_state_dict': model.state_dict(),
            'best_params': best_params,
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': best_vloss,
            'train_mae': avg_train_mae,
            'val_mae': best_vmae,
            'bucket_stats': best_buckets,
            'many_mae': many_mae,
            'few_mae': few_mae,
            'best_view': best_view,
            'config': CONFIG
        }
        if scheduler is not None:
            checkpoint_dict['scheduler_state_dict'] = scheduler.state_dict()
        torch.save(checkpoint_dict, CONFIG['CHECKPOINT_NAME'])
        
        if passes_constraint:
            print(f" ✅ BEST ({best_view}) - PASSES CONSTRAINT!")
        else:
            print(f" ✅ BEST ({best_view})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve > CONFIG['EARLY_STOP_PATIENCE']:
            print(f"\n\nEarly stopping at epoch {epoch+1}")
            break
        else:
            print("")

print(f"\nTraining complete")
print(f"Best Epoch: {best_epoch+1}/{len(train_curve)}")
print(f"Best Val Loss: {best_loss:.5f}")
if best_epoch >= 0:
    print(f"Bucket MAE: Many={bucket_many[best_epoch]:.5f} | "
          f"Med={bucket_medium[best_epoch]:.5f} | Few={bucket_few[best_epoch]:.5f}")

Epoch   1/300 | TrL: 2.11116 | VaL: 1.39773 | TrMAE: 1.22103 | VaMAE: 1.41657 | Many: 1.4537 | Med: 1.1414 | Few: 0.8861 | LR: 0.000300 | LIVE ✅ BEST (LIVE)
Epoch   2/300 | TrL: 1.26493 | VaL: 1.28678 | TrMAE: 1.34549 | VaMAE: 1.15605 | Many: 1.1807 | Med: 0.8256 | Few: 1.2538 | LR: 0.000300 | LIVE ✅ BEST (LIVE)
Epoch   3/300 | TrL: 1.22968 | VaL: 1.17033 | TrMAE: 1.28698 | VaMAE: 1.24939 | Many: 1.2929 | Med: 0.8446 | Few: 0.8894 | LR: 0.000300 | LIVE ✅ BEST (LIVE)
Epoch   4/300 | TrL: 1.17513 | VaL: 1.13436 | TrMAE: 1.29574 | VaMAE: 1.03004 | Many: 1.0554 | Med: 0.7108 | Few: 1.0699 | LR: 0.000300 | EMA ✅ BEST (EMA)
Epoch   5/300 | TrL: 1.21015 | VaL: 1.10780 | TrMAE: 1.29747 | VaMAE: 1.03837 | Many: 1.0677 | Med: 0.6875 | Few: 1.0294 | LR: 0.000300 | EMA ✅ BEST (EMA)
Epoch   6/300 | TrL: 1.07246 | VaL: 1.01503 | TrMAE: 1.32890 | VaMAE: 1.08642 | Many: 1.1256 | Med: 0.7446 | Few: 0.6906 | LR: 0.000300 | LIVE ✅ BEST (LIVE)
Epoch   7/300 | TrL: 1.10544 | VaL: 1.01047 | TrMAE: 1.27611 |

KeyboardInterrupt: 

## Load Best Model

In [ ]:
checkpoint = torch.load(CONFIG['CHECKPOINT_NAME'], map_location=device)

if 'best_params' in checkpoint:
    state_dict = model.state_dict()
    named_params = [n for n, _ in model.named_parameters()]
    restored = {name: tensor.to(device) for name, tensor in zip(named_params, checkpoint['best_params'])}
    state_dict.update(restored)
    model.load_state_dict(state_dict)
else:
    model.load_state_dict(checkpoint['model_state_dict'])

model.to(device).eval()

print(f"Best model loaded from epoch {checkpoint['epoch']+1}")
print(f"Val Loss: {checkpoint['val_loss']:.5f} | Val MAE: {checkpoint['val_mae']:.5f}")
print(f"Many: {checkpoint['many_mae']:.4f}° | Few: {checkpoint['few_mae']:.4f}°")

## Training Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=CONFIG['FIG_SIZE'])

# Loss curves
axes[0, 0].plot(train_curve, label="Train", color='blue', linewidth=2)
axes[0, 0].plot(val_curve, label="Validation", color='orange', linewidth=2)
axes[0, 0].axvline(x=best_epoch, color='red', linestyle='--', linewidth=2, label='Best')
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].set_title("Training & Validation Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# MAE curves
axes[0, 1].plot(train_mae_curve, label="Train MAE", color='green', linewidth=2)
axes[0, 1].plot(val_mae_curve, label="Val MAE", color='red', linewidth=2)
axes[0, 1].axvline(x=best_epoch, color='red', linestyle='--', linewidth=2, label='Best')
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("MAE (°)")
axes[0, 1].set_title("Mean Absolute Error")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Bucket MAE
axes[1, 0].plot(bucket_many, label="Many-shot", color='blue', linewidth=2)
axes[1, 0].plot(bucket_medium, label="Medium-shot", color='orange', linewidth=2)
axes[1, 0].plot(bucket_few, label="Few-shot", color='red', linewidth=2)
axes[1, 0].axvline(x=best_epoch, color='red', linestyle='--', linewidth=2)
axes[1, 0].axhline(y=CONFIG['MANY_SHOT_CONSTRAINT'], color='black', linestyle=':', 
                   linewidth=2, label=f"Constraint ({CONFIG['MANY_SHOT_CONSTRAINT']}°)")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Bucket MAE (°)")
axes[1, 0].set_title("Bucket-wise Performance")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Learning rate
axes[1, 1].plot(learning_rate_curve, color='purple', linewidth=2, marker='o', markersize=3)
axes[1, 1].axvline(x=best_epoch, color='red', linestyle='--', linewidth=2, label='Best')
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Learning Rate")
axes[1, 1].set_title("Learning Rate Schedule")
axes[1, 1].set_yscale('log')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTraining Summary:")
print(f"Best Epoch: {best_epoch + 1}/{len(train_curve)}")
print(f"Best Val Loss: {min(val_curve):.5f}")
print(f"Final LR: {learning_rate_curve[-1]:.6f}")
print(f"\nBucket MAE at Best Epoch:")
print(f"  Many-shot:   {bucket_many[best_epoch]:.4f}°")
print(f"  Medium-shot: {bucket_medium[best_epoch]:.4f}°")
print(f"  Few-shot:    {bucket_few[best_epoch]:.4f}°")

## Training Data Visualization (Plotly)

In [ ]:
print("Loading unshuffled training data for visualization...")
train_dataloader_unshuffled = load_test_data(
    test_path=CONFIG['TRAIN_PATH'],
    batchsize=CONFIG['BATCH_SIZE'],
    pipeline=pipeline,
    seed=CONFIG['SEED'],
    load_anonymized_data=False
)

print(f"Batches: {len(train_dataloader_unshuffled)}")

In [ ]:
# Generate predictions on unshuffled training data
model.eval()
train_preds = []
train_ground_truths = []

with torch.no_grad():
    for x, y in train_dataloader_unshuffled:
        x = x.to(device)
        pred = model(x)
        train_preds.append(pred.cpu().numpy())
        train_ground_truths.append(y.cpu().numpy())

train_preds = np.concatenate(train_preds).flatten()
train_ground_truths = np.concatenate(train_ground_truths).flatten()

train_mae = np.abs(train_preds - train_ground_truths).mean()
print(f"Training Data MAE: {train_mae:.5f}°")
print(f"Predictions shape: {train_preds.shape}")

In [ ]:
# Interactive Plotly visualization of training predictions
fig = go.Figure()

fig.add_trace(go.Scatter(
    y=train_ground_truths,
    mode='lines',
    name='Ground Truth',
    line=dict(color='blue', width=1)
))

fig.add_trace(go.Scatter(
    y=train_preds,
    mode='lines',
    name='Predictions',
    line=dict(color='red', width=1),
    opacity=0.8
))

fig.update_layout(
    title=f'Training Data: Ground Truth vs Predictions (MAE: {train_mae:.5f}°)',
    xaxis_title='Sample Index',
    yaxis_title='Sideslip Angle (°)',
    width=CONFIG['PLOTLY_WIDTH'],
    height=CONFIG['PLOTLY_HEIGHT'],
    hovermode='x unified'
)

fig.show()

## Kaggle Metric Validation

In [ ]:
# Prepare data for Kaggle metric
dist_df = pd.read_csv(CONFIG['DIST_CSV'])
shot_band_mapping = dist_df.groupby("sideSlip")["band"].first()

# Create ground truth DataFrame with band labels
gt_df = pd.DataFrame({
    "ground_truth_sideslip": train_ground_truths
}, dtype=np.float64)
gt_df["id"] = range(len(train_ground_truths))
gt_df["band"] = gt_df["ground_truth_sideslip"].round(1).map(shot_band_mapping).fillna("unknown")

# Create predictions DataFrame
preds_df = pd.DataFrame({
    "id": range(len(train_preds)),
    "predicted_sideslip": train_preds
})

# Calculate Kaggle metric
print("Calculating Kaggle metric on training data...")
print("=" * 80)
kaggle_score = score(gt_df, preds_df, row_id_column_name="id")
print("=" * 80)

print(f"\n📊 Band Distribution:")
print(gt_df["band"].value_counts())

print(f"\n🎯 Training Data Kaggle Score: {kaggle_score}")

## Generate Test Predictions

In [ ]:
test_dataloader = load_test_data(
    test_path=CONFIG['TEST_PATH'],
    batchsize=CONFIG['BATCH_SIZE'],
    pipeline=pipeline,
    seed=CONFIG['SEED'],
    load_anonymized_data=True
)

model.eval()
test_preds = []

with torch.no_grad():
    for x, y in test_dataloader:
        x = x.to(device)
        pred = model(x)
        test_preds.append(pred.cpu().numpy())

test_preds = np.concatenate(test_preds).flatten()

print(f"Generated {len(test_preds):,} predictions")
print(f"Expected: 49,328 samples for Kaggle submission")
print(f"Range: [{test_preds.min():.2f}°, {test_preds.max():.2f}°]")
print(f"Mean: {test_preds.mean():.3f}° | Std: {test_preds.std():.3f}°")

if len(test_preds) == 49328:
    print("✅ Correct number of predictions!")
else:
    print(f"❌ WARNING: Expected 49,328 but got {len(test_preds)}")

## Save Submission

In [ ]:
y_pred = pd.DataFrame({
    "id": range(len(test_preds)),
    "predicted_sideslip": test_preds
})

y_pred.to_csv(CONFIG['SUBMISSION_CSV'], index=False)

print(f"Predictions saved to: {CONFIG['SUBMISSION_CSV']}")
print(f"Total samples: {len(test_preds):,}")
print("Ready for submission!")